# 🌿 Sampoorn Kisan AI Sahayak — Disease Detection Agent

This notebook trains the **Plant Disease Detection Agent** for the multi-agent farming assistant.

**What it does:**
1. Loads the leaf-image dataset (38 classes, PlantVillage-style folder structure) from Google Drive
2. Trains an image classifier using transfer learning (MobileNetV2)
3. Fine-tunes the model for better accuracy
4. Wraps the trained model in a `disease_agent()` function that:
   - Takes a leaf image
   - Returns the predicted disease, confidence score
   - Returns symptoms, treatment, and prevention advice (from a built-in knowledge base)
5. Saves everything so it can be plugged into the rest of the multi-agent system (crop agent, weather agent, orchestrator, etc.) as a callable tool

**Before running:** upload your dataset folder to Google Drive (the one shown in your screenshots — 38 subfolders like `Tomato___Late_blight`, `Apple___healthy`, etc.) and update `DATASET_DIR` in the cell below.

## 1. Setup

In [ ]:
# Mount Google Drive (skip this cell if running outside Colab)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, callbacks
import numpy as np
import matplotlib.pyplot as plt
import os, json

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# ---- EDIT THIS ----
# Path to the folder that directly contains the 38 class subfolders
# (Apple___Apple_scab, Apple___Black_rot, ..., Tomato___Tomato_Yellow_Leaf_Curl_Virus)
DATASET_DIR = "/content/drive/MyDrive/PlantVillage"   # <-- change to your actual path

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


## 2. Data pipeline

Uses an 80/20 train-validation split straight from the folder structure, with augmentation on the training set only (real leaf photos from farmers won't be perfectly centered/lit, so augmentation helps generalization).

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED
)

val_gen = val_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False
)

NUM_CLASSES = train_gen.num_classes
CLASS_INDICES = train_gen.class_indices
IDX_TO_CLASS = {v: k for k, v in CLASS_INDICES.items()}
print(f"Classes found: {NUM_CLASSES}")
print(CLASS_INDICES)


## 3. Build the model

Transfer learning with **MobileNetV2** — small and fast enough to eventually run on a low-end phone/server, which matters for a farmer-facing app. The base is frozen first, then partially unfrozen for fine-tuning.

In [ ]:
base_model = MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze for the first training phase

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


## 4. Phase 1 — train the classification head (base frozen)

In [ ]:
checkpoint_path = "/content/drive/MyDrive/disease_agent_model.keras"

cb = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    callbacks.ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]

history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=cb
)


## 5. Phase 2 — fine-tune

Unfreeze the top layers of MobileNetV2 and continue training with a much lower learning rate so the pretrained weights adjust gently instead of being wrecked.

In [ ]:
base_model.trainable = True

# Freeze everything except the last ~30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=cb
)


## 6. Evaluate

In [ ]:
def plot_history(h1, h2):
    acc = h1.history['accuracy'] + h2.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss'] + h2.history['loss']
    val_loss = h1.history['val_loss'] + h2.history['val_loss']

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(acc, label='train')
    plt.plot(val_acc, label='val')
    plt.title('Accuracy'); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(loss, label='train')
    plt.plot(val_loss, label='val')
    plt.title('Loss'); plt.legend()
    plt.show()

plot_history(history1, history2)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

val_gen.reset()
preds = model.predict(val_gen, verbose=1)
y_pred = np.argmax(preds, axis=1)
y_true = val_gen.classes

print(classification_report(y_true, y_pred, target_names=list(CLASS_INDICES.keys())))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()


## 7. Save the model + class mapping

In [ ]:
model.save(checkpoint_path)

with open("/content/drive/MyDrive/class_indices.json", "w") as f:
    json.dump(IDX_TO_CLASS, f, indent=2)

print("Saved model to:", checkpoint_path)
print("Saved class mapping to: /content/drive/MyDrive/class_indices.json")


## 8. Disease knowledge base

Maps each class label to plain-language symptoms, treatment, and prevention advice, so the agent doesn't just output a class name — it outputs something a farmer can actually act on. Extend/correct any entry with local agronomic guidance as needed.

In [ ]:
DISEASE_INFO = {
  "Apple___Apple_scab": {"disease": "Apple Scab", "symptoms": "Olive-green to black velvety spots on leaves and fruit; leaves may yellow and drop early.", "treatment": "Remove and destroy fallen leaves; apply a fungicide (e.g. captan or myclobutanil) starting at bud break.", "prevention": "Choose scab-resistant apple varieties; prune for airflow; avoid overhead irrigation."},
  "Apple___Black_rot": {"disease": "Apple Black Rot", "symptoms": "Purple-bordered brown leaf spots ('frogeye leaf spot'); rotting, mummified fruit.", "treatment": "Prune out cankers and dead wood; apply fungicide during the growing season; remove mummified fruit.", "prevention": "Sanitize pruning tools; remove nearby dead/diseased wood promptly."},
  "Apple___Cedar_apple_rust": {"disease": "Cedar Apple Rust", "symptoms": "Bright orange-yellow spots on leaves; can cause premature leaf drop and fruit blemishes.", "treatment": "Apply fungicide from pink bud stage through a few weeks after petal fall.", "prevention": "Remove nearby juniper/cedar trees if possible (alternate host); plant resistant varieties."},
  "Apple___healthy": {"disease": "Healthy", "symptoms": "No visible disease symptoms.", "treatment": "None needed.", "prevention": "Continue regular monitoring, balanced fertilization, and proper irrigation."},
  "Blueberry___healthy": {"disease": "Healthy", "symptoms": "No visible disease symptoms.", "treatment": "None needed.", "prevention": "Maintain acidic, well-drained soil and regular monitoring."},
  "Cherry_(including_sour)___healthy": {"disease": "Healthy", "symptoms": "No visible disease symptoms.", "treatment": "None needed.", "prevention": "Regular monitoring and proper pruning for airflow."},
  "Cherry_(including_sour)___Powdery_mildew": {"disease": "Powdery Mildew", "symptoms": "White powdery fungal growth on leaves and shoots; leaf curling and stunted growth.", "treatment": "Apply sulfur-based or potassium bicarbonate fungicide; remove heavily infected shoots.", "prevention": "Prune for good air circulation; avoid excess nitrogen fertilizer."},
  "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": {"disease": "Gray Leaf Spot", "symptoms": "Rectangular gray-tan lesions on leaves running parallel to veins.", "treatment": "Apply a foliar fungicide if disease pressure is high; especially in continuous corn fields.", "prevention": "Rotate crops; use resistant hybrids; manage crop residue (tillage)."},
  "Corn_(maize)___Common_rust": {"disease": "Common Rust", "symptoms": "Small, reddish-brown, powdery pustules scattered on both leaf surfaces.", "treatment": "Fungicide application if severe and detected early; usually not economically necessary in resistant hybrids.", "prevention": "Plant rust-resistant hybrids; monitor early in the season."},
  "Corn_(maize)___healthy": {"disease": "Healthy", "symptoms": "No visible disease symptoms.", "treatment": "None needed.", "prevention": "Continue regular monitoring and balanced fertilization."},
  "Corn_(maize)___Northern_Leaf_Blight": {"disease": "Northern Leaf Blight", "symptoms": "Long, elliptical, grayish-green to tan lesions on leaves.", "treatment": "Apply fungicide at early disease onset; especially under humid conditions.", "prevention": "Rotate crops; use resistant hybrids; till under crop residue."},
  "Grape___Black_rot": {"disease": "Grape Black Rot", "symptoms": "Small brown circular leaf spots; fruit shrivels into hard black 'mummies'.", "treatment": "Apply fungicide from early shoot growth through fruit set; remove mummified berries.", "prevention": "Prune for airflow; remove infected plant debris; canopy management."},
  "Grape___Esca_(Black_Measles)": {"disease": "Esca (Black Measles)", "symptoms": "Tiger-stripe pattern of yellow/red-brown streaks between leaf veins; dark spots on berries.", "treatment": "No reliable cure — remove and destroy severely infected vines/wood; protect pruning wounds.", "prevention": "Avoid pruning in wet weather; disinfect tools; use trunk renewal on infected vines."},
  "Grape___healthy": {"disease": "Healthy", "symptoms": "No visible disease symptoms.", "treatment": "None needed.", "prevention": "Continue regular monitoring and canopy management."},
  "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)": {"disease": "Grape Leaf Blight", "symptoms": "Dark brown angular spots on leaves, often with yellow halos; leaves may dry and drop.", "treatment": "Apply copper-based or other labeled fungicide.", "prevention": "Improve air circulation; remove fallen infected leaves; avoid overhead watering."},
  "Orange___Haunglongbing_(Citrus_greening)": {"disease": "Citrus Greening (HLB)", "symptoms": "Blotchy mottled yellowing of leaves, lopsided bitter fruit, twig dieback.", "treatment": "No cure — remove and destroy infected trees to slow spread; control the Asian citrus psyllid vector with insecticide.", "prevention": "Use certified disease-free planting material; monitor and control psyllid populations."},
  "Peach___Bacterial_spot": {"disease": "Bacterial Spot", "symptoms": "Small dark water-soaked spots on leaves and fruit that may become sunken/cracked on fruit.", "treatment": "Apply copper-based bactericide during dormant season and early growing season.", "prevention": "Plant resistant varieties; avoid overhead irrigation; prune for airflow."},
  "Peach___healthy": {"disease": "Healthy", "symptoms": "No visible disease symptoms.", "treatment": "None needed.", "prevention": "Continue regular monitoring and balanced fertilization."},
  "Pepper,_bell___Bacterial_spot": {"disease": "Bacterial Spot", "symptoms": "Small water-soaked spots on leaves turning brown with yellow halo; scabby spots on fruit.", "treatment": "Apply copper-based bactericide; remove severely infected plants.", "prevention": "Use disease-free seed/transplants; avoid overhead watering; rotate crops."},
  "Pepper,_bell___healthy": {"disease": "Healthy", "symptoms": "No visible disease symptoms.", "treatment": "None needed.", "prevention": "Continue regular monitoring and balanced fertilization."},
  "Potato___Early_blight": {"disease": "Early Blight", "symptoms": "Dark concentric 'target-ring' spots on older/lower leaves first.", "treatment": "Apply fungicide (e.g. chlorothalonil or mancozeb); remove infected foliage.", "prevention": "Rotate crops; avoid overhead irrigation; ensure balanced fertilization (avoid nitrogen stress)."},
  "Potato___healthy": {"disease": "Healthy", "symptoms": "No visible disease symptoms.", "treatment": "None needed.", "prevention": "Continue regular monitoring and proper irrigation."},
  "Potato___Late_blight": {"disease": "Late Blight", "symptoms": "Large dark, water-soaked lesions on leaves, often with white fungal growth on leaf undersides in humid weather; can destroy a crop fast.", "treatment": "Apply fungicide immediately (e.g. mancozeb or metalaxyl-based); remove and destroy infected plants.", "prevention": "Use certified disease-free seed potatoes; avoid overhead irrigation; monitor closely during cool, wet weather."},
  "Raspberry___healthy": {"disease": "Healthy", "symptoms": "No visible disease symptoms.", "treatment": "None needed.", "prevention": "Continue regular monitoring and proper pruning."},
  "Soybean___healthy": {"disease": "Healthy", "symptoms": "No visible disease symptoms.", "treatment": "None needed.", "prevention": "Continue regular monitoring and crop rotation."},
  "Squash___Powdery_mildew": {"disease": "Powdery Mildew", "symptoms": "White powdery patches on leaves and stems; leaves may yellow and dry out.", "treatment": "Apply sulfur or potassium bicarbonate-based fungicide; remove heavily infected leaves.", "prevention": "Space plants for airflow; avoid overhead watering; choose resistant varieties."},
  "Strawberry___healthy": {"disease": "Healthy", "symptoms": "No visible disease symptoms.", "treatment": "None needed.", "prevention": "Continue regular monitoring and proper irrigation."},
  "Strawberry___Leaf_scorch": {"disease": "Leaf Scorch", "symptoms": "Small purple spots on leaves that merge into larger scorched-looking blotches.", "treatment": "Apply fungicide; remove infected leaves after harvest.", "prevention": "Avoid overhead watering; space plants for airflow; remove old infected foliage."},
  "Tomato___Bacterial_spot": {"disease": "Bacterial Spot", "symptoms": "Small dark water-soaked spots on leaves and fruit, often with yellow halo.", "treatment": "Apply copper-based bactericide; remove severely infected plants.", "prevention": "Use disease-free seed/transplants; avoid overhead watering; rotate crops."},
  "Tomato___Early_blight": {"disease": "Early Blight", "symptoms": "Dark concentric 'target-ring' spots on older leaves first, often with yellowing around them.", "treatment": "Apply fungicide (e.g. chlorothalonil or mancozeb); remove infected lower leaves.", "prevention": "Rotate crops; mulch to reduce soil splash; stake plants for airflow."},
  "Tomato___healthy": {"disease": "Healthy", "symptoms": "No visible disease symptoms.", "treatment": "None needed.", "prevention": "Continue regular monitoring and balanced fertilization."},
  "Tomato___Late_blight": {"disease": "Late Blight", "symptoms": "Large, irregular, water-soaked greasy-looking lesions on leaves and stems; can spread and kill plants rapidly in cool, wet weather.", "treatment": "Apply fungicide immediately; remove and destroy infected plants to stop spread.", "prevention": "Avoid overhead irrigation; ensure airflow; monitor closely during humid weather; use resistant varieties."},
  "Tomato___Leaf_Mold": {"disease": "Leaf Mold", "symptoms": "Pale green/yellow spots on upper leaf surface with olive-green to grayish mold underneath.", "treatment": "Improve ventilation (especially greenhouse); apply fungicide if severe.", "prevention": "Reduce humidity; avoid overhead watering; space plants well."},
  "Tomato___Septoria_leaf_spot": {"disease": "Septoria Leaf Spot", "symptoms": "Small circular spots with dark borders and gray/tan centers, mainly on lower leaves.", "treatment": "Apply fungicide; remove and destroy infected lower leaves.", "prevention": "Rotate crops; avoid overhead watering; mulch to reduce soil splash."},
  "Tomato___Spider_mites Two-spotted_spider_mite": {"disease": "Two-Spotted Spider Mite Infestation", "symptoms": "Fine yellow/white stippling on leaves, fine webbing, leaves may bronze and dry out.", "treatment": "Apply miticide or insecticidal soap/neem oil; increase humidity to discourage mites.", "prevention": "Avoid drought stress; monitor regularly, especially in hot dry weather."},
  "Tomato___Target_Spot": {"disease": "Target Spot", "symptoms": "Brown lesions with concentric rings, resembling a target, on leaves, stems and fruit.", "treatment": "Apply fungicide; remove severely infected foliage.", "prevention": "Ensure good airflow; avoid overhead watering; rotate crops."},
  "Tomato___Tomato_mosaic_virus": {"disease": "Tomato Mosaic Virus", "symptoms": "Mottled light/dark green mosaic pattern on leaves; stunted, distorted growth.", "treatment": "No cure — remove and destroy infected plants; disinfect tools and hands.", "prevention": "Use certified virus-free seed; control weeds; wash hands/tools between plants; avoid tobacco use near plants."},
  "Tomato___Tomato_Yellow_Leaf_Curl_Virus": {"disease": "Tomato Yellow Leaf Curl Virus", "symptoms": "Upward curling, yellowing leaves; stunted plants; reduced fruit set.", "treatment": "No cure — remove and destroy infected plants; control whitefly vector with insecticide/traps.", "prevention": "Use resistant varieties; control whiteflies; use reflective mulch or insect netting."}
}

print(f"Knowledge base entries: {len(DISEASE_INFO)} (dataset classes: {NUM_CLASSES})")


## 9. The Disease Agent function

This is the piece that plugs into the rest of the multi-agent system. It takes an image path (or PIL image), runs the model, and returns a structured response the orchestrator agent (or a farmer-facing chat UI) can use directly.

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image

def disease_agent(img_path, model=model, idx_to_class=IDX_TO_CLASS, top_k=1, confidence_threshold=0.4):
    """
    The Plant Disease Detection Agent.

    Args:
        img_path: path to a leaf image (or a PIL.Image object)
        model: trained Keras model
        idx_to_class: dict mapping class index -> class label
        top_k: how many top predictions to return
        confidence_threshold: below this, the agent flags low confidence

    Returns:
        dict — structured agent output, ready to be returned by an API
        or consumed by an orchestrator/LLM agent.
    """
    if isinstance(img_path, str):
        img = keras_image.load_img(img_path, target_size=IMG_SIZE)
    else:
        img = img_path.resize(IMG_SIZE)

    arr = keras_image.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    arr = preprocess_input(arr)

    preds = model.predict(arr, verbose=0)[0]
    top_indices = preds.argsort()[-top_k:][::-1]

    results = []
    for idx in top_indices:
        label = idx_to_class[idx]
        confidence = float(preds[idx])
        info = DISEASE_INFO.get(label, {
            "disease": label, "symptoms": "Not documented.",
            "treatment": "Not documented.", "prevention": "Not documented."
        })
        results.append({
            "class_label": label,
            "disease": info["disease"],
            "confidence": round(confidence, 4),
            "symptoms": info["symptoms"],
            "treatment": info["treatment"],
            "prevention": info["prevention"]
        })

    top_confidence = results[0]["confidence"]
    response = {
        "agent": "disease_detection_agent",
        "status": "low_confidence" if top_confidence < confidence_threshold else "ok",
        "predictions": results
    }
    if response["status"] == "low_confidence":
        response["message"] = (
            "Confidence is low — ask the farmer for a clearer, well-lit close-up photo of the "
            "affected leaf, or confirm which crop this is."
        )
    return response


## 10. Test it

In [ ]:
# Point this at any leaf image (upload a test image to Colab or use one from the val set)
test_image_path = "/content/drive/MyDrive/sample_leaf.jpg"  # <-- change this

result = disease_agent(test_image_path)
print(json.dumps(result, indent=2))


## 11. Next steps — wiring this into the multi-agent system

This notebook gives you a trained model + `disease_agent()` function. To match the architecture in your slides (multi-agent, RAG-based, orchestrator-driven):

1. **Export as a service**: wrap `disease_agent()` in a FastAPI/Flask endpoint (e.g. `POST /diagnose` accepting an image) so the orchestrator agent can call it as a tool, the same way it would call the crop-recommendation or weather agent.
2. **Expose it as an LLM tool/function**: define a JSON schema for `disease_agent` (input: image; output: disease, confidence, treatment, prevention) so your orchestrator LLM can invoke it via function calling, then pass the structured result into the response-generation agent for a natural-language reply to the farmer.
3. **RAG layer (optional, matches your base paper)**: instead of (or in addition to) the static `DISEASE_INFO` dict, retrieve treatment guidance from an agricultural knowledge base/vector store, so advice can be regionally tailored and kept up to date.
4. **Confidence handling**: the `low_confidence` status above is meant to trigger the orchestrator to ask a clarifying follow-up (e.g. "could you send a clearer photo?") rather than confidently guessing.
5. **Model export for deployment**: for a lighter mobile/edge deployment later, convert the saved `.keras` model to TensorFlow Lite (`tf.lite.TFLiteConverter`).

If you'd like, I can build the FastAPI wrapper or the LangChain/function-calling tool definition next.